# Current loop: the motor driver between the controller and the robot

The motors' current settles in about 1 ms ($L_m / R$) and a slipping wheel changes speed within a few ms, much faster than a 50 Hz controller can react. Holding a voltage for 20 ms lets the current drift as the back-EMF changes, so it can pass any limit the controller set. A real DC motor driver closes a fast current loop instead: the controller asks for a current, and the driver sets the voltage every plant step to deliver it.

```text
controller (50 Hz)  --current targets-->  current loop (1 kHz)  --voltage-->  robot (1 kHz)
       ^                                        ^                                 |
       |                                        +------ current, wheel speed -----+
       +---------------- measure: [X, Y, theta, omega_1..4] ----------------------+
```

* **Robot:** unchanged. It is loaded from `RobotSimulationOpenLoopModel_Merged.ipynb` by cell id, so there is one plant.
* **Current loop:** sets the voltage from the current target, the measured current and the wheel speed, every $\Delta T = 1$ ms. The per-wheel current limit is enforced here, every step.
* **Controller:** asks for currents every $T_s = 20$ ms and still sees only `measure`. The tests below use fixed current targets; no MPC yet.

In [ ]:
# ===========================================================================
# The robot: constants and plant cells of the merged notebook
# ===========================================================================
import json
import numpy as np
import matplotlib.pyplot as plt

PLANT_NOTEBOOK = "RobotSimulationOpenLoopModel_Merged.ipynb"
plant_cells = {c.get("id"): "".join(c["source"]) for c in json.load(open(PLANT_NOTEBOOK, encoding="utf-8"))["cells"]}
exec(plant_cells["constants"])     # simulation_parameters, GRAVITY
exec(plant_cells["plant"])         # OmniMotorState, OmniRobotSimulation
print(f"Loaded the robot from {PLANT_NOTEBOOK}: dt = {simulation_parameters['dt'] * 1e3:.0f} ms")

## The current loop

Each wheel's motor obeys $L_m \frac{di}{dt} = V - R\,i - K_b\,\omega$. The loop sets the voltage every step $k$ from the current target $i^{ref}$, the measured current $i_k$ and the wheel speed $\omega_k$:

$$i^{*}_{k} = \text{clip}\left(i^{ref}_k,\ -I_{max},\ I_{max}\right)$$
$$V_k = \text{clip}\Big(\underbrace{R\, i^{*}_k + K_b\, \omega_k}_{\text{feedforward}} + \underbrace{K_p\,(i^{*}_k - i_k) + K_i \sum_{j \le k} (i^{*}_j - i_j)\,\Delta T}_{\text{feedback}},\ -V_{max},\ V_{max}\Big)$$

* **Feedforward:** the voltage that holds $i^{*}$ at the present wheel speed: the resistive drop plus the back-EMF. It is updated every step, so the voltage follows the back-EMF as the wheel speeds up or slows down.
* **Feedback:** $K_p$ closes the remaining error within a few steps; $K_i$ removes any steady error. The integral stops while the voltage is clipped (anti-windup).
* **Current limit:** the target is clipped to $\pm I_{max}$ every step, with $I_{max,i} = \mu_{limit} F_{n,i,\text{static}}\, r / K_t$, the current whose torque uses $\mu_{limit}$ of the wheel's static grip.
* **Supply limit:** at high wheel speed the back-EMF leaves too little of $V_{max}$ to reach the target, and the current falls short. That is the motor's real top-speed limit, not a fault of the loop.

**Choosing $K_p$:** with the robot's backward Euler motor update and the wheel speed held over one step, the current error obeys $e_{k+1} = \frac{L_m/\Delta T - K_p}{L_m/\Delta T + R}\, e_k$. $K_p = L_m/\Delta T = 2.3\ \Omega$ would close it in one step; half that, $K_p = 1.15\ \Omega$, leaves $0.26$ of the error after each step (about 1 ms to settle) with margin for a real motor's parameters being off.

In [ ]:
"""
Current loop
============

CurrentLoop is the motor driver: created once from simulation_parameters and current_loop_parameters,
it returns the four voltages for one plant step from the current targets, the measured currents and
the wheel speeds.
"""

current_loop_parameters = {
    'Kp': 1.15,          # Proportional gain on the current error (V/A): half of Lm/dt
    'Ki': 100.0,         # Integral gain on the current error (V/(A s))
    'mu_limit': 0.7,     # Share of each wheel's static grip its current limit allows
}


class CurrentLoop:
    """
    The 1 kHz current loop of the four motor drivers.
    """

    def __init__(self, params: dict, loop_params: dict):
        """
        params: simulation_parameters (motor values, supply limit, wheel loads)
        loop_params: current_loop_parameters
        """
        self.p = params
        self.Kp = loop_params['Kp']
        self.Ki = loop_params['Ki']

        # Per-wheel current limit: the current whose torque uses mu_limit of the static grip
        geometry = OmniRobotSimulation(params)          # fixed values only: the static wheel loads
        F_n_static = np.array([geometry.F_n_front_static] * 2 + [geometry.F_n_rear_static] * 2)
        self.I_max = loop_params['mu_limit'] * F_n_static * params['r'] / params['Kt']

        self.integral = np.zeros(4)                     # integral part of the voltage [V]

    def voltage(self, I_ref: np.ndarray, I: np.ndarray, omega: np.ndarray) -> np.ndarray:
        """
        I_ref: current targets from the controller [A]
        I: measured motor currents [A]
        omega: measured wheel speeds [rad/s]

        Returns the four voltages for this plant step [V].
        """
        R, Kb, V_max, dt = self.p['R'], self.p['Kb'], self.p['V_max'], self.p['dt']

        # Current limit, every step
        I_target = np.clip(I_ref, -self.I_max, self.I_max)
        error = I_target - I

        # Feedforward (resistive drop and back-EMF at the present speed), then feedback
        V = R * I_target + Kb * omega + self.Kp * error + self.integral
        V_out = np.clip(V, -V_max, V_max)

        # Integrate only where the voltage is not clipped (anti-windup)
        free = V == V_out
        self.integral[free] += self.Ki * dt * error[free]
        return V_out

In [ ]:
"""
Runs
====

run_current_loop: the controller's current targets are held for Ts, and the current loop sets the voltage
every plant step. run_voltage_hold, for comparison: every Ts the voltage that gives the target current at
that moment's wheel speed, V = R I + Kb omega, is set and held for Ts, as the MPC did so far.
"""


def run_current_loop(I_ref_of_t, T: float, params: dict, loop_params: dict, Ts: float = 0.02, random_seed: int = 42) -> dict:
    """
    I_ref_of_t: current targets as a function of time [A] (what a controller would ask for)
    T: run length [s]
    Ts: controller period [s]: the targets change only every Ts

    Returns a dictionary of arrays at every plant step: time, I_ref (after the limit), I, V, omega, slip [m/s], pose.
    """
    robot = OmniRobotSimulation(params, random_seed=random_seed)
    loop = CurrentLoop(params, loop_params)
    return _run(robot, T, params, Ts, I_ref_of_t,
                lambda I_ref, state: loop.voltage(I_ref, state.I, state.omega), loop.I_max)


def run_voltage_hold(I_ref_of_t, T: float, params: dict, loop_params: dict, Ts: float = 0.02, random_seed: int = 42) -> dict:
    """The same targets, but the voltage is worked out once every Ts and held."""
    robot = OmniRobotSimulation(params, random_seed=random_seed)
    I_max = CurrentLoop(params, loop_params).I_max
    held = {'V': np.zeros(4)}

    def voltage(I_ref, state, k):
        if k % int(round(Ts / params['dt'])) == 0:
            held['V'] = np.clip(params['R'] * I_ref + params['Kb'] * state.omega, -params['V_max'], params['V_max'])
        return held['V']
    return _run(robot, T, params, Ts, I_ref_of_t, voltage, I_max, pass_step=True)


def _run(robot, T, params, Ts, I_ref_of_t, voltage, I_max, pass_step=False):
    state = OmniMotorState()
    dt = params['dt']
    hold = int(round(Ts / dt))
    steps = int(round(T / dt))
    log = {name: np.zeros((steps, size)) for name, size in
           (('I_ref', 4), ('I', 4), ('V', 4), ('omega', 4), ('slip', 4), ('pose', 3))}
    log['time'] = np.arange(steps) * dt
    log['I_max'] = I_max
    for k in range(steps):
        if k % hold == 0:
            I_ref = np.clip(I_ref_of_t(k * dt), -I_max, I_max)     # controller rate
        V = voltage(I_ref, state, k) if pass_step else voltage(I_ref, state)
        log['I_ref'][k], log['I'][k], log['V'][k] = I_ref, state.I, V
        v_center_d = robot.H_drive @ state.v_local
        state, _, _ = robot.step(state, V)
        log['omega'][k] = state.omega
        log['slip'][k] = v_center_d - params['r'] * state.omega
        log['pose'][k] = state.pos_global
    return log


def summary(log: dict) -> str:
    """Current tracking (from 5 ms after each new target), largest current against the limit, voltage changes."""
    settled = (np.arange(len(log['time'])) % 20) >= 5          # skip the first 5 ms after each new target
    error = np.abs(log['I'] - log['I_ref'])[settled]
    over = np.max(np.abs(log['I']) / log['I_max'])
    steps_per_hold = 20
    V = log['V'][:len(log['V']) // steps_per_hold * steps_per_hold].reshape(-1, steps_per_hold, 4)
    spread = np.mean(np.max(V, axis=1) - np.min(V, axis=1))
    return (f"current error: RMS {1e3 * np.sqrt(np.mean(error**2)):.0f} mA, largest {1e3 * error.max():.0f} mA | "
            f"largest current {over:.2f} x limit | voltage range within each 20 ms: {spread:.2f} V on average | "
            f"slip over 5 mm/s: {100 * np.mean(np.any(np.abs(log['slip']) > 5e-3, axis=1)):.0f} % of the time")

## Tests

Noise is off so only the loop is tested. Current targets use the wheel sign patterns of Dana's tests (forward, strafe, rotate).

1. **Step response:** a 0.4 A forward target from rest. How fast does the current reach it, and does the voltage keep changing as the wheels speed up?
2. **Current loop against a held voltage:** a manoeuvre with forward, reverse, strafe and turning targets held for 20 ms at a time, as a controller would send them. The targets are wheel currents that add up to a chassis force without the wheels pushing against each other (with four wheels and three directions of motion, other combinations make the wheels fight and slip). With the voltage held, the current drifts as the back-EMF changes; with the loop it should stay on target and inside the limit.
3. **Supply limit:** a full-limit forward target held until the wheels near their top speed. The current must fall short once the back-EMF uses up the 12 V, and the integral must not wind up: when the target drops to braking, the current follows without overshoot.

In [ ]:
# ===========================================================================
# Test 1: step response
# ===========================================================================
QUIET = dict(simulation_parameters, sigma_V=0.0, sigma_mu=0.0)
FWD = np.array([1.0, -1.0, -1.0, 1.0])
STR = np.array([-1.0, -1.0, 1.0, 1.0])
ROT = np.ones(4)

log = run_current_loop(lambda t: 0.4 * FWD, 0.3, QUIET, current_loop_parameters)
within = np.abs(log['I'] - log['I_ref']) < 0.02 * np.abs(log['I_ref'])     # within 2 % of the target
settle_ms = np.argmax(within, axis=0) * QUIET['dt'] * 1e3
print(f"Current within 2 % of the 0.4 A target after {settle_ms} ms (wheels 1-4)")
print("Test 1 | " + summary(log))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for i in range(4):
    axes[0].plot(1e3 * log['time'][:30], log['I'][:30, i], label=f'wheel {i + 1}')
    axes[1].plot(log['time'], log['V'][:, i], label=f'wheel {i + 1}')
axes[0].axhline(0.4, color='k', ls='--'); axes[0].axhline(-0.4, color='k', ls='--')
axes[0].set_title('Current, first 30 ms'); axes[0].set_xlabel('time [ms]'); axes[0].set_ylabel('I [A]'); axes[0].legend(); axes[0].grid(True)
axes[1].set_title('Voltage set by the loop: follows the back-EMF as the wheels speed up'); axes[1].set_xlabel('time [s]'); axes[1].set_ylabel('V [V]'); axes[1].grid(True)
plt.tight_layout(); plt.show()

In [ ]:
# ===========================================================================
# Test 2: current loop against a held voltage
# ===========================================================================
H = OmniRobotSimulation(QUIET).H_drive


def wheel_currents(F_local):
    """
    Current targets [A] that push the chassis with F_local = [force right, force forward, moment] (N, N, N m).

    Four wheels and three directions of motion leave one combination of wheel forces that only makes the
    wheels push against each other. H (H^T H)^-1 gives the wheel forces with none of it, and I = r F / Kt.
    """
    return QUIET['r'] / QUIET['Kt'] * H @ np.linalg.solve(H.T @ H, F_local)


def manoeuvre(t):
    """Forward, reverse, strafe, then strafe back while turning, 0.5 s each (chassis forces of about 2 N)."""
    if t < 0.5:
        return wheel_currents([0.0, 2.0, 0.0])
    if t < 1.0:
        return wheel_currents([0.0, -2.0, 0.0])
    if t < 1.5:
        return wheel_currents([2.0, 0.0, 0.0])
    return wheel_currents([-2.0, 0.0, 0.05])

with_loop = run_current_loop(manoeuvre, 2.0, QUIET, current_loop_parameters)
with_hold = run_voltage_hold(manoeuvre, 2.0, QUIET, current_loop_parameters)
print("Current loop  | " + summary(with_loop))
print("Voltage held  | " + summary(with_hold))

fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=True)
for col, (name, log) in enumerate((("Current loop (1 kHz)", with_loop), ("Voltage held for 20 ms", with_hold))):
    for i in range(4):
        axes[0, col].plot(log['time'], log['I'][:, i], color=f'C{i}', label=f'wheel {i + 1}')
        axes[0, col].plot(log['time'], log['I_ref'][:, i], color=f'C{i}', ls='--', lw=0.8)
        axes[1, col].plot(log['time'], log['V'][:, i], color=f'C{i}')
    for i in range(4):
        axes[0, col].axhline(log['I_max'][i], color=f'C{i}', ls=':', lw=0.6)
        axes[0, col].axhline(-log['I_max'][i], color=f'C{i}', ls=':', lw=0.6)
    axes[0, col].set_title(f'{name}: current (dashed: target, dotted: limit)'); axes[0, col].set_ylabel('I [A]'); axes[0, col].grid(True)
    axes[1, col].set_title(f'{name}: voltage'); axes[1, col].set_xlabel('time [s]'); axes[1, col].set_ylabel('V [V]'); axes[1, col].grid(True)
axes[0, 0].legend()
plt.tight_layout(); plt.show()

In [ ]:
# ===========================================================================
# Test 3: supply limit and anti-windup
# ===========================================================================
def full_then_drop(t):
    """Full-limit forward target for 3 s, then a 0.4 A braking target (reachable at top speed)."""
    return (10.0 if t < 3.0 else -0.4) * FWD     # 10 A is clipped to each wheel's limit

log = run_current_loop(full_then_drop, 3.5, QUIET, current_loop_parameters)
k_drop = int(3.0 / QUIET['dt'])
short = np.abs(log['I_ref'][:k_drop]) - np.abs(log['I'][:k_drop])
print(f"Top wheel speed reached: {np.max(np.abs(log['omega'])):.1f} rad/s (12 V / Kb = {QUIET['V_max'] / QUIET['Kb']:.1f} rad/s at no load)")
print(f"Current short of the target before the drop: up to {1e3 * short.max():.0f} mA (supply limit)")
after = log['I'][k_drop:k_drop + 50] - log['I_ref'][k_drop:k_drop + 50]
print(f"After the target drops to braking: largest overshoot past it {1e3 * np.max(np.sign(log['I_ref'][k_drop]) * after):.0f} mA, "
      f"error from 5 ms on {1e3 * np.max(np.abs(after[5:])):.0f} mA (no wind-up if small)")
print("Test 3 | " + summary(log))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for i in range(4):
    axes[0].plot(log['time'], log['I'][:, i], color=f'C{i}', label=f'wheel {i + 1}')
    axes[0].plot(log['time'], log['I_ref'][:, i], color=f'C{i}', ls='--', lw=0.8)
    axes[1].plot(log['time'], log['V'][:, i], color=f'C{i}')
    axes[2].plot(log['time'], log['omega'][:, i], color=f'C{i}')
axes[0].set_title('Current (dashed: target)'); axes[0].set_ylabel('I [A]'); axes[0].legend(); axes[0].grid(True)
axes[1].set_title('Voltage: reaches the 12 V supply'); axes[1].set_ylabel('V [V]'); axes[1].grid(True)
axes[2].set_title('Wheel speed'); axes[2].set_ylabel('omega [rad/s]'); axes[2].grid(True)
for ax in axes: ax.set_xlabel('time [s]')
plt.tight_layout(); plt.show()